# OmniSpeak (Colab)

Đọc văn bản, nhân bản giọng nói, lưu thư viện giọng — chạy trên Google Colab, dùng model [OmniVoice](https://github.com/k2-fsa/OmniVoice) (`k2-fsa/OmniVoice`, Apache-2.0).

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy các cell theo thứ tự từ trên xuống — mọi cell đều an toàn khi chạy lại.


## Cài đặt

In [ ]:
# 1. GPU check
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Không có GPU — Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu.")


In [ ]:
# 2. Cài đặt
import subprocess
import sys

def run(cmd, what=""):
    print("\n$", cmd if isinstance(cmd, str) else " ".join(cmd))
    if subprocess.run(cmd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"Lỗi: {what or cmd}. Xem log phía trên rồi chạy lại cell này.")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1")
run([sys.executable, "-m", "pip", "install", "-q",
     "omnivoice", "fastapi", "uvicorn[standard]", "python-multipart", "soundfile"])
run([sys.executable, "-c",
     "from omnivoice import OmniVoice, VoiceClonePrompt; import fastapi, soundfile; print('OK')"])


In [ ]:
# 3. Giao diện web — tải index.html từ GitHub thay vì nhúng trong notebook
import os
import time
import urllib.request
import urllib.error

FRONTEND_DIR = "/content/omnispeak_frontend"
os.makedirs(FRONTEND_DIR, exist_ok=True)

GITHUB_USER = "trkhanh8312-make"
GITHUB_REPO = "omnispeak"
GITHUB_BRANCH = "main"
GITHUB_PATH = "frontend/index.html"  # đường dẫn file trong repo, vd "frontend/index.html"

# thêm ?_=<timestamp> để né cache CDN của raw.githubusercontent.com (~5 phút),
# đảm bảo luôn lấy đúng bản mới nhất bạn vừa commit
FRONTEND_URL = (
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/"
    f"{GITHUB_BRANCH}/{GITHUB_PATH}?_={int(time.time())}"
)

dest = os.path.join(FRONTEND_DIR, "index.html")
try:
    urllib.request.urlretrieve(FRONTEND_URL, dest)
    size = os.path.getsize(dest)
    if size < 500:  # file quá nhỏ -> nhiều khả năng GitHub trả về trang lỗi 404 dạng HTML ngắn
        raise RuntimeError(f"File tải về chỉ {size} byte — có vẻ không phải index.html thật.")
    print(f"Đã tải giao diện web từ GitHub ({size} byte).")
except (urllib.error.URLError, urllib.error.HTTPError, RuntimeError) as e:
    raise RuntimeError(
        "Không tải được giao diện web từ GitHub.\n"
        f"URL đã thử: {FRONTEND_URL}\n"
        "Kiểm tra lại:\n"
        "  1) File đã commit & push lên GitHub chưa?\n"
        "  2) Đường dẫn GITHUB_PATH ở đầu cell này có đúng không?\n"
        "  3) Repo có để public không (raw.githubusercontent.com không đọc được repo private)?\n"
        f"Lỗi gốc: {e}"
    )


In [ ]:
# 4. Mount Google Drive (tuỳ chọn) — lưu bền vững giọng nói + model đã tải
import os

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/omnispeak_data"
HF_DIR = "/content/drive/MyDrive/omnispeak_hf_cache"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(HF_DIR, exist_ok=True)
os.environ["OMNISPEAK_DATA_DIR"] = DATA_DIR
os.environ["HF_HOME"] = HF_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"  # Drive (FUSE) không hỗ trợ symlink
print("Giọng nói + model sẽ lưu bền vững trên Drive.")

In [ ]:
# 5. Tải trước model (bỏ qua cũng được — sẽ tự tải khi khởi động backend)
from huggingface_hub import snapshot_download

print("Model tại:", snapshot_download("k2-fsa/OmniVoice"))


In [ ]:
# 6. Backend API — FastAPI gọi thẳng OmniVoice
import os

APP_DIR = "/content/omnispeak_app"
os.makedirs(APP_DIR, exist_ok=True)

BACKEND_PY = '''\
import io
import json
import os
import time
import uuid
from pathlib import Path
from typing import Optional

import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, Form, UploadFile, File, HTTPException
from fastapi.responses import Response
from fastapi.staticfiles import StaticFiles
from omnivoice import OmniVoice, VoiceClonePrompt

DATA_DIR = Path(os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data"))
PROFILES_DIR = DATA_DIR / "profiles"
INDEX_PATH = DATA_DIR / "profiles.json"
FRONTEND_DIR = os.environ.get("OMNISPEAK_FRONTEND_DIR", "/content/omnispeak_frontend")
PROFILES_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
MODEL = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype, load_asr=True)


def _load_index():
    return json.loads(INDEX_PATH.read_text()) if INDEX_PATH.exists() else []


def _save_index(items):
    INDEX_PATH.write_text(json.dumps(items, ensure_ascii=False, indent=2))


@app.get("/health")
def health():
    return {"status": "ok", "device": "cuda" if torch.cuda.is_available() else "cpu"}


@app.post("/generate")
async def generate(text: str = Form(...), profile_id: Optional[str] = Form(None)):
    t0 = time.time()
    kwargs = {}
    if profile_id:
        p = PROFILES_DIR / f"{profile_id}.pt"
        if not p.exists():
            raise HTTPException(404, f"Profile {profile_id} not found")
        kwargs["voice_clone_prompt"] = VoiceClonePrompt.load(str(p))

    audio = MODEL.generate(text=text, **kwargs)
    gen_time = time.time() - t0

    # model.generate() có thể trả numpy hoặc torch — chuẩn hoá về numpy mono
    wav = audio[0]
    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().float().numpy()
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.T
        if wav.shape[1] == 1:
            wav = wav[:, 0]

    buf = io.BytesIO()
    sf.write(buf, wav, 24000, format="WAV", subtype="PCM_16")
    return Response(content=buf.getvalue(), media_type="audio/wav",
                     headers={"X-Gen-Time": f"{gen_time:.2f}"})


@app.get("/profiles")
def list_profiles():
    return _load_index()


@app.post("/profiles")
async def create_profile(name: str = Form(...), kind: str = Form("clone"),
                          ref_audio: UploadFile = File(...), ref_text: Optional[str] = Form(None)):
    items = _load_index()
    existing = next((p for p in items if p["name"] == name), None)
    if existing:
        return existing

    profile_id = uuid.uuid4().hex[:12]
    tmp = DATA_DIR / f"_upload_{profile_id}.wav"
    tmp.write_bytes(await ref_audio.read())
    try:
        prompt = MODEL.create_voice_clone_prompt(ref_audio=str(tmp), ref_text=ref_text or None)
        prompt.save(str(PROFILES_DIR / f"{profile_id}.pt"))
    finally:
        tmp.unlink(missing_ok=True)

    entry = {"id": profile_id, "name": name, "kind": kind}
    items.append(entry)
    _save_index(items)
    return entry


@app.delete("/profiles/{profile_id}")
def delete_profile(profile_id: str):
    items = _load_index()
    remaining = [p for p in items if p["id"] != profile_id]
    if len(remaining) == len(items):
        raise HTTPException(404, "Not found")
    _save_index(remaining)
    (PROFILES_DIR / f"{profile_id}.pt").unlink(missing_ok=True)
    return {"deleted": profile_id}


app.mount("/", StaticFiles(directory=FRONTEND_DIR, html=True), name="frontend")
'''

with open(os.path.join(APP_DIR, "backend.py"), "w", encoding="utf-8") as f:
    f.write(BACKEND_PY)
print("Đã ghi backend.py.")


In [ ]:
# 7. Khởi động backend
FORCE_RESTART = True  # luôn nạp lại code mới nhất từ cell 6

import json
import os
import subprocess
import sys
import time
import urllib.request

APP_DIR = "/content/omnispeak_app"
PORT = 3900
LOG_PATH = "/content/omnispeak_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = None if FORCE_RESTART else health()
if info:
    print("Backend đang chạy —", info)
else:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    env["OMNISPEAK_DATA_DIR"] = os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data")
    env["OMNISPEAK_FRONTEND_DIR"] = "/content/omnispeak_frontend"
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend:app", "--app-dir", APP_DIR,
         "--host", "127.0.0.1", "--port", str(PORT)],
        env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Đang khởi động (PID {proc.pid})...")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print("Backend đã sẵn sàng —", info)
    else:
        try:
            tail = "".join(open(LOG_PATH, errors="replace").readlines()[-40:])
        except OSError:
            tail = "(không có log)"
        raise SystemExit(f"Backend không lên được sau 5 phút.\n--- log ---\n{tail}")


In [ ]:
# 8. Mở giao diện web
from google.colab import output

output.serve_kernel_port_as_window(3900)


### Tổng kết

Danh sách giọng nói đã lưu.

In [ ]:
# Tổng kết
import requests

try:
    profiles = requests.get("http://127.0.0.1:3900/profiles", timeout=15).json()
    print(f"Giọng đã lưu ({len(profiles)}):")
    for p in profiles:
        print(" ", p["id"], p.get("name"))
except Exception as e:
    print("Không lấy được danh sách:", e)


## Xử lý sự cố

- **`device: cpu` hoặc generate chậm** — bật GPU: Runtime → Change runtime type → T4 GPU.
- **Sửa `backend.py` (cell 6) xong mà lỗi vẫn y nguyên** — cell 7 có `FORCE_RESTART = True`, chạy lại cell 7 là tự nạp code mới, không cần tự kill process.
- **Muốn giữ giọng nói + model qua các phiên sau** — chạy cell 4 (mount Drive) trước cell 5.
- **Tab UI trắng hoặc lỗi** — chạy lại cell 7 rồi cell 8. Cho phép pop-up cho `colab.research.google.com`; hoặc đổi cell 8 sang `output.serve_kernel_port_as_iframe(3900)` để nhúng UI ngay trong notebook.
- **Xem log backend** — `/content/omnispeak_backend.log`.
